# Nightjar Composition

This notebook assembles complete Nightjar takes from a cached local clip pool. The main composition path does not run RAVE: it reuses decoded bird clips and AI-generated Nightjar pool clips already stored in this folder.

## Setup

The arrangement is fixed: birds, original Nightjar, AI-generated Nightjar, birds. The AI section has four route modes: close, free, wild, and outer loop.

In [ ]:
from pathlib import Path
import json
import math
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import librosa
import soundfile as sf
from IPython.display import HTML, display

ROOT = Path.cwd()
PAYLOAD_JSON = ROOT / "data" / "nightjar_composition_payload.json"
OUTPUT_ROOT = ROOT / "outputs"
COMPOSITION_DIR = OUTPUT_ROOT / "compositions"
FIGURE_DIR = OUTPUT_ROOT / "figures"
TABLE_DIR = OUTPUT_ROOT / "tables"

source_from_env = os.environ.get("NIGHTJAR_SOURCE_WAV", "")
SOURCE_NIGHTJAR = Path(source_from_env).expanduser() if source_from_env else None
if SOURCE_NIGHTJAR is None:
    candidates = [
        ROOT / "data" / "nightjar.wav",
        Path("/Users/v/projects/Nightjar_project/nightjar_rave/data/nj_music_generation_data/nightjar_song/nightjar.wav"),
    ]
    SOURCE_NIGHTJAR = next((p for p in candidates if p.exists()), candidates[0])

SR = 44_100
BIRD_SECONDS = 36.0
AI_SECONDS = 72.0
CLIP_CROSSFADE_SECONDS = 0.35
SECTION_CROSSFADE_SECONDS = 2.5
TARGET_RMS = 0.10
ORIGINAL_RMS = 0.11
PEAK_LIMIT = 0.98
TAKES = [
    {"id": "close", "name": "close to Nightjar", "seed": 77},
    {"id": "free", "name": "free route", "seed": 91},
    {"id": "wild", "name": "wild route", "seed": 123},
    {"id": "outer", "name": "outer loop", "seed": 177},
]

for path in (COMPOSITION_DIR, FIGURE_DIR, TABLE_DIR):
    path.mkdir(parents=True, exist_ok=True)

print("payload:", PAYLOAD_JSON)
print("source:", SOURCE_NIGHTJAR)

## Cached route data

The payload lists the local bird clips, Nightjar anchors, and AI-generated Nightjar pool clips. Bird clips are used only at the beginning and end of each composition.

In [ ]:
if not PAYLOAD_JSON.exists():
    raise FileNotFoundError(f"Missing cached payload: {PAYLOAD_JSON}")
if not SOURCE_NIGHTJAR.exists():
    raise FileNotFoundError(f"Missing original Nightjar wav: {SOURCE_NIGHTJAR}")

DATA = json.loads(PAYLOAD_JSON.read_text(encoding="utf-8"))
anchors = pd.DataFrame(DATA["anchors"])
nodes = pd.DataFrame(DATA["nodes"])
bird_nodes = pd.DataFrame(DATA["birdNodes"])

print(f"anchors: {len(anchors)}")
print(f"AI Nightjar clips: {len(nodes)}")
print(f"bird clips: {len(bird_nodes)}")

assert DATA["meta"]["sr"] == SR
assert len(anchors) == 49
assert len(nodes) == 539
assert len(bird_nodes) == 48

referenced = list(nodes["audio"]) + list(bird_nodes["audio"])
missing = [path for path in referenced if not (ROOT / path).exists()]
if missing:
    raise FileNotFoundError(f"Missing local clip assets: {missing[:5]}")

anchor_xyz = anchors[["x", "y", "z"]].to_numpy(np.float32)
node_xyz = nodes[["x", "y", "z"]].to_numpy(np.float32)
node_anchor_xyz = anchors.set_index("i").loc[nodes["anchor"], ["x", "y", "z"]].to_numpy(np.float32)
anchor_center = np.median(anchor_xyz, axis=0)

nodes["anchor_distance"] = np.linalg.norm(node_xyz - node_anchor_xyz, axis=1)
nodes["cloud_distance"] = np.linalg.norm(node_xyz - anchor_center, axis=1)
nodes["wide"] = nodes["variant"].isin(DATA["meta"].get("wideVariants", []))
nodes["far_score"] = (
    0.62 * nodes["anchor_distance"].rank(pct=True)
    + 0.28 * nodes["cloud_distance"].rank(pct=True)
    + 0.10 * nodes["wide"].astype(float)
)


## Generation reference

The composition path uses cached clips. Regenerating that cache is a separate RAVE step, not part of this notebook run.

In [ ]:
# Reference only. Keep this disabled for composition runs.
#
# Cache rebuild outline:
# 1. Encode 3-second Nightjar and bird windows with the trained RAVE model.
# 2. Decode several near-anchor latent variants for each window.
# 3. Save the decoded clips under audio/ and write their coordinates to data/nightjar_composition_payload.json.
#
# This notebook expects that decoded cache to already exist.

## Audio helpers

In [ ]:
def normalize(y, rms=TARGET_RMS, peak=PEAK_LIMIT):
    y = np.nan_to_num(np.asarray(y, np.float32))
    if y.size == 0:
        return y
    y = y - float(np.mean(y))
    level = float(np.sqrt(np.mean(y * y)) + 1e-12)
    if level > 1e-10:
        y = y * (rms / level)
    top = float(np.max(np.abs(y)) + 1e-12)
    if top > peak:
        y = y * (peak / top)
    return y.astype(np.float32)


def load_audio(path):
    y, _ = librosa.load(path, sr=SR, mono=True)
    return np.asarray(y, np.float32)


def fit_duration(y, seconds):
    target = int(round(seconds * SR))
    y = np.asarray(y, np.float32)
    if len(y) >= target:
        return y[:target]
    return np.pad(y, (0, target - len(y))).astype(np.float32)


def equal_power_join(parts, fade_seconds):
    parts = [np.asarray(p, np.float32) for p in parts if len(p)]
    if not parts:
        return np.zeros(0, np.float32)
    fade = int(round(fade_seconds * SR))
    out = parts[0]
    for part in parts[1:]:
        n = min(fade, len(out), len(part))
        if n <= 0:
            out = np.concatenate([out, part]).astype(np.float32)
            continue
        t = np.linspace(0, 1, n, dtype=np.float32)
        cross = out[-n:] * np.cos(t * np.pi / 2) + part[:n] * np.sin(t * np.pi / 2)
        out = np.concatenate([out[:-n], cross, part[n:]]).astype(np.float32)
    return out


def clip_count(seconds):
    clip = float(DATA["meta"]["clipSec"])
    stride = max(0.25, clip - CLIP_CROSSFADE_SECONDS)
    return max(1, math.ceil(max(0, seconds - clip) / stride) + 1)


def render_records(records, target_seconds=None):
    parts = [normalize(load_audio(ROOT / rec["audio"])) for rec in records]
    y = equal_power_join(parts, CLIP_CROSSFADE_SECONDS)
    if target_seconds is not None:
        y = fit_duration(y, target_seconds)
    return normalize(y)

## Route builders

The bird outro mirrors the intro in reverse order. The AI routes use only Nightjar pool clips; birds are not used inside the AI section.

In [ ]:
def pick_row(frame, rng):
    return frame.iloc[int(rng.integers(len(frame)))].to_dict()


def circular_distance(a, b, count):
    d = abs(int(a) - int(b))
    return min(d, count - d)


def point_distance(a, b):
    return float(np.linalg.norm(np.array([a["x"], a["y"], a["z"]]) - np.array([b["x"], b["y"], b["z"]])))


def bird_phrase(seconds, rng):
    anchor_order = sorted(bird_nodes["anchor"].unique())
    offset = int(rng.integers(len(anchor_order)))
    records = []
    for i in range(clip_count(seconds)):
        anchor = anchor_order[(offset + i) % len(anchor_order)]
        candidates = bird_nodes[bird_nodes["anchor"] == anchor]
        records.append(pick_row(candidates, rng))
    return records


def weighted_choice(items, weights, rng):
    weights = np.asarray(weights, np.float64)
    weights = np.maximum(weights, 1e-12)
    weights = weights / weights.sum()
    return items[int(rng.choice(len(items), p=weights))]


def mode_steps(mode):
    if mode == "close":
        return np.array([1, 0, 2, -1]), np.array([0.68, 0.16, 0.10, 0.06])
    if mode == "free":
        return np.array([1, -1, 0, 2, -2, 3, -3, 6, -6]), np.array([0.18, 0.16, 0.14, 0.12, 0.11, 0.09, 0.08, 0.06, 0.06])
    if mode == "wild":
        return np.array([0, 2, -2, 3, -3, 6, -6, 9, -9, 12, -12]), np.array([0.08, 0.10, 0.10, 0.10, 0.10, 0.13, 0.13, 0.10, 0.08, 0.05, 0.03])
    raise ValueError(mode)


def next_ai_node(current, mode, step_index, total_steps, rng, recent_ids, recent_anchors):
    count = len(anchors)
    steps, step_weights = mode_steps(mode)
    step = weighted_choice(steps, step_weights, rng)
    target_anchor = int((current["anchor"] + step) % count)
    candidates = nodes[nodes["anchor"] == target_anchor].to_dict("records")

    wide_variants = set(DATA["meta"].get("wideVariants", []))
    weights = []
    for candidate in candidates:
        novelty = 0.06 if candidate["id"] in recent_ids else 1.0
        anchor_novelty = 0.40 if candidate["anchor"] in recent_anchors else 1.0
        if mode == "close":
            variant_shape = 2.4 / (1 + candidate["variant"])
            wide_shape = 0.18 if candidate["variant"] in wide_variants else 1.0
            far_shape = 1.0 / (1.0 + 2.0 * candidate["far_score"])
        elif mode == "free":
            variant_shape = 1.0 + min(candidate["variant"], 8) * 0.05
            wide_shape = 1.10 if candidate["variant"] in wide_variants else 1.0
            far_shape = 0.75 + candidate["far_score"]
        else:
            variant_shape = 0.65 + min(candidate["variant"], 10) * 0.12
            wide_shape = 2.35 if candidate["variant"] in wide_variants else 0.85
            far_shape = 0.50 + 2.5 * candidate["far_score"]
        weights.append(novelty * anchor_novelty * variant_shape * wide_shape * far_shape)
    return weighted_choice(candidates, weights, rng)


def outer_candidates(current, progress, home_anchor, recent_ids):
    count = len(anchors)
    frame = nodes.copy()
    if progress < 0.25:
        frame = frame[frame["far_score"] >= frame["far_score"].quantile(0.78)]
    elif progress < 0.72:
        frame = frame[frame["far_score"] >= frame["far_score"].quantile(0.86)]
    else:
        radius = max(2, int((1.0 - progress) * 32))
        frame = frame[frame["anchor"].map(lambda a: circular_distance(a, home_anchor, count) <= radius)]
        if progress > 0.88:
            frame = frame[frame["variant"] <= 3]
    if len(frame) < 8:
        frame = nodes
    return frame[~frame["id"].isin(recent_ids)].to_dict("records") or frame.to_dict("records")


def outer_route(seconds, rng):
    steps = clip_count(seconds)
    home_anchor = int(anchors["i"].max())
    current = pick_row(nodes[(nodes["anchor"] == home_anchor) & (nodes["variant"] <= 3)], rng)
    route = [current]
    for step_index in range(1, steps):
        progress = step_index / max(1, steps - 1)
        recent = route[-10:]
        recent_ids = {item["id"] for item in recent}
        candidates = outer_candidates(current, progress, home_anchor, recent_ids)
        weights = []
        for candidate in candidates:
            d = point_distance(current, candidate)
            smooth = np.exp(-((d - 1.7) ** 2) / (2 * 1.15 ** 2)) + 0.10
            novelty = 0.05 if candidate["id"] in recent_ids else 1.0
            outward = 0.35 + 3.2 * candidate["far_score"]
            if progress > 0.72:
                home = 1.0 / (1.0 + circular_distance(candidate["anchor"], home_anchor, len(anchors)))
                outward = 0.45 + candidate["far_score"]
                weights.append(smooth * novelty * outward * (0.35 + 8.0 * home))
            else:
                weights.append(smooth * novelty * outward)
        current = weighted_choice(candidates, weights, rng)
        route.append(current)
    return route


def ai_route(seconds, mode, rng):
    if mode == "outer":
        return outer_route(seconds, rng)
    steps = clip_count(seconds)
    start_anchor = int(anchors["i"].max())
    current = pick_row(nodes[nodes["anchor"] == start_anchor], rng)
    route = [current]
    for step_index in range(1, steps):
        recent = route[-10:]
        recent_ids = {item["id"] for item in recent}
        recent_anchors = {item["anchor"] for item in recent[-6:]}
        current = next_ai_node(current, mode, step_index, steps, rng, recent_ids, recent_anchors)
        route.append(current)
    return route

## Compose takes

In [ ]:
def build_take(take):
    rng = np.random.default_rng(take["seed"])
    intro_records = bird_phrase(BIRD_SECONDS, rng)
    outro_records = list(reversed(intro_records))
    ai_records = ai_route(AI_SECONDS, take["id"], rng)

    original = normalize(load_audio(SOURCE_NIGHTJAR), rms=ORIGINAL_RMS)
    sections = [
        ("birds_intro", render_records(intro_records, BIRD_SECONDS)),
        ("nightjar_original", original),
        ("ai_generated", render_records(ai_records, AI_SECONDS)),
        ("birds_outro", render_records(outro_records, BIRD_SECONDS)),
    ]
    audio = normalize(equal_power_join([audio for _, audio in sections], SECTION_CROSSFADE_SECONDS))

    output_path = COMPOSITION_DIR / f"nightjar_composition_{take['id']}.wav"
    sf.write(output_path, audio, SR)

    rows = []
    order = 0
    for section, records in [
        ("birds_intro", intro_records),
        ("ai_generated", ai_records),
        ("birds_outro", outro_records),
    ]:
        for record in records:
            rows.append({
                "take": take["id"],
                "take_name": take["name"],
                "seed": take["seed"],
                "order": order,
                "section": section,
                "clip_id": record.get("id"),
                "anchor": record.get("anchor"),
                "variant": record.get("variant"),
                "audio": record.get("audio"),
                "x": record.get("x"),
                "y": record.get("y"),
                "z": record.get("z"),
                "rms": record.get("rms"),
                "centroid": record.get("centroid"),
                "anchor_distance": record.get("anchor_distance"),
                "cloud_distance": record.get("cloud_distance"),
                "far_score": record.get("far_score"),
                "wide": record.get("wide"),
            })
            order += 1
    return output_path, audio, pd.DataFrame(rows), ai_records, len(audio) / SR


all_rows = []
take_routes = []
outputs = []
for take in TAKES:
    path, audio, rows, ai_records, duration = build_take(take)
    rows["composition"] = path.name
    all_rows.append(rows)
    take_routes.append({"take": take, "route": ai_records})
    outputs.append(path)
    print(f"{take['name']}: {path.name}, {duration:.1f}s")
    display(HTML(f'<audio controls src="{path.relative_to(ROOT).as_posix()}"></audio>'))

routes_df = pd.concat(all_rows, ignore_index=True)
routes_path = TABLE_DIR / "nightjar_composition_routes.csv"
routes_df.to_csv(routes_path, index=False)

original_seconds = librosa.get_duration(path=str(SOURCE_NIGHTJAR))
timing_rows = [
    {"section": "birds_intro", "starts_at": 0.0, "note": "opening bird phrase"},
    {"section": "nightjar_original", "starts_at": BIRD_SECONDS - SECTION_CROSSFADE_SECONDS, "note": "original Nightjar fades in"},
    {"section": "ai_generated", "starts_at": BIRD_SECONDS + original_seconds - 2 * SECTION_CROSSFADE_SECONDS, "note": "AI-generated Nightjar pool route fades in"},
    {"section": "birds_outro", "starts_at": BIRD_SECONDS + original_seconds + AI_SECONDS - 3 * SECTION_CROSSFADE_SECONDS, "note": "bird phrase returns in reverse"},
]
timing_df = pd.DataFrame(timing_rows)
timing_df["timecode"] = timing_df["starts_at"].map(lambda s: f"{int(s // 60)}:{int(round(s % 60)):02d}")
timing_path = TABLE_DIR / "nightjar_composition_timing.csv"
timing_df.to_csv(timing_path, index=False)
display(timing_df)
routes_df.head()

## Route view

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1.25, 1]})
ax = axes[0]
ax.scatter(nodes["x"], nodes["y"], s=9, alpha=0.10, color="#d94b4b", label="AI pool")
ax.scatter(bird_nodes["x"], bird_nodes["y"], s=18, alpha=0.45, color="#f0a11a", label="birds")
ax.plot(anchors["x"], anchors["y"], color="#171717", linewidth=1.5, alpha=0.65, label="Nightjar anchors")
ax.scatter(anchors["x"], anchors["y"], s=16, color="#171717", alpha=0.75)

colors = {"close": "#2878bd", "free": "#36a852", "wild": "#e5484d", "outer": "#7a55c7"}
for item in take_routes:
    route_df = pd.DataFrame(item["route"])
    take = item["take"]
    ax.plot(route_df["x"], route_df["y"], marker="o", markersize=3, linewidth=2, color=colors[take["id"]], label=take["name"])

ax.set_title("AI routes")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.grid(True, alpha=0.25)
ax.legend(loc="best", fontsize=8)

timeline = axes[1]
original_seconds = librosa.get_duration(path=str(SOURCE_NIGHTJAR))
section_lengths = [
    ("birds", BIRD_SECONDS, "#f0a11a"),
    ("original", original_seconds, "#171717"),
    ("AI", AI_SECONDS, "#2878bd"),
    ("birds", BIRD_SECONDS, "#f0a11a"),
]
start = 0
for label, seconds, color in section_lengths:
    timeline.barh([0], [seconds], left=[start], color=color, alpha=0.86)
    timeline.text(start + seconds / 2, 0, label, ha="center", va="center", color="white", fontsize=9, fontweight="bold")
    start += seconds

timeline.set_title("Composition sections")
timeline.set_yticks([])
timeline.set_xlabel("seconds")
timeline.set_xlim(0, start)
timeline.grid(True, axis="x", alpha=0.25)

fig.tight_layout()
figure_path = FIGURE_DIR / "nightjar_composition_routes.png"
fig.savefig(figure_path, dpi=160)
figure_path

## Checks

In [ ]:
check_rows = []
for path in outputs:
    y, sr = sf.read(path, always_2d=False)
    y = np.asarray(y, np.float32)
    check_rows.append({
        "file": path.name,
        "sr": sr,
        "duration_seconds": len(y) / sr,
        "rms": float(np.sqrt(np.mean(y * y))),
        "peak": float(np.max(np.abs(y))),
        "finite": bool(np.isfinite(y).all()),
    })

checks = pd.DataFrame(check_rows)
display(checks)
assert (checks["sr"] == SR).all()
assert checks["finite"].all()
assert (checks["rms"] > 1e-4).all()
assert (checks["peak"] <= PEAK_LIMIT + 1e-4).all()
assert len(routes_df) > 0
print("ok")